## Validation of aggergates

In [2]:
import os

os.chdir("..")
import pandas as pd

from src.ingestion.database import get_engine

engine = get_engine()

print("✓ Connected")

✓ Connected


In [3]:
aggregate_tables = [
    "agg_daily_sales",
    "agg_product_performance",
    "agg_customer_performance",
    "agg_seller_performance",
    "agg_geography_performance"
]

results = []

for table in aggregate_tables:

    count = pd.read_sql(
        f"SELECT COUNT(*) AS row_count FROM {table}",
        engine
    )["row_count"].iloc[0]

    results.append({
        "table": table,
        "row_count": count
    })

pd.DataFrame(results)

,table,row_count
0,agg_daily_sales,616
1,agg_product_performance,32951
2,agg_customer_performance,98666
3,agg_seller_performance,3095
4,agg_geography_performance,27


### revenue reconciliation


In [4]:
pd.read_sql("""
SELECT

    ROUND(
        (SELECT SUM(total_item_value)
         FROM fact_sales),
        2
    ) AS fact_revenue,

    ROUND(
        (SELECT SUM(total_revenue)
         FROM agg_customer_performance),
        2
    ) AS customer_aggregate_revenue,

    ROUND(
        (SELECT SUM(total_sales_value)
         FROM agg_daily_sales),
        2
    ) AS daily_aggregate_revenue;
""", engine)

,fact_revenue,customer_aggregate_revenue,daily_aggregate_revenue
0,15843553.24,15843553.24,15843553.24


### Product revenue reconciliation

In [5]:
pd.read_sql("""
SELECT

    ROUND(
        (SELECT SUM(total_item_value)
         FROM fact_sales),
        2
    ) AS fact_revenue,

    ROUND(
        (SELECT SUM(revenue)
         FROM agg_product_performance),
        2
    ) AS product_revenue,

    ROUND(
        (SELECT SUM(revenue)
         FROM agg_product_performance)
        -
        (SELECT SUM(total_item_value)
         FROM fact_sales),
        2
    ) AS difference;
""", engine)

,fact_revenue,product_revenue,difference
0,15843553.24,15843553.24,0.0


### Seller revenue reconciliation

In [6]:
pd.read_sql("""
SELECT

    ROUND(
        (SELECT SUM(total_item_value)
         FROM fact_sales),
        2
    ) AS fact_revenue,

    ROUND(
        (SELECT SUM(total_revenue)
         FROM agg_seller_performance),
        2
    ) AS seller_revenue;
""", engine)

,fact_revenue,seller_revenue
0,15843553.24,15843553.24


### Customer aggregate validation

In [7]:
pd.read_sql("""
SELECT
    COUNT(*) AS customers,

    SUM(order_count) AS total_orders,

    SUM(item_count) AS total_items,

    ROUND(
        SUM(total_revenue),
        2
    ) AS total_revenue

FROM agg_customer_performance;
""", engine)

,customers,total_orders,total_items,total_revenue
0,98666,98666.0,112650.0,15843553.24
